# AI 音乐工具的疗愈音频生成体验

由于我本人是 ADHD 患者，也是 Spotify Focus Playlist 的重度用户，因此我对改善专注力的功能性音乐有切身的需求理解，并做了一定的研究。本任务中，我将跳过 LLM 用户需求拆解，直接从我自身的体验和研究出发，以循证的方式构建 prompt，利用 [Suno](https://suno.com/) 和 [ACE-Step](https://acemusic.ai/) 两种模型生成 Focus Music，并对比二者的优缺点。

## ADHD 是一种怎样的体验？

ADHD，学名”注意力缺陷与多动障碍“，是一种全球高发的神经发育障碍。ADHD 在我国儿童中的患病率为 $6.4\%$。ADHD 分为两种类型：

- **注意力缺陷型 (ADHD-I)**：主要表现为难以维持注意力、易分心、工作记忆缺失。
- **多动冲动型 (ADHD-HI)**：主要表现为过度活动（如坐立不安、小动作多）、说话过多、难以等待、常打断他人。

我本人是 ADHD-I 患者。在学习和工作过程中，我最大的困扰是工作启动慢，很难进入心流状态。上课时，如果我觉得课程无聊，我的大脑会离开课程，产生各种”随机“思维，这些思维非常纷杂，并且会在多个主题间切换。在外人看来，我此时的状态像做”白日梦“一样。而当我想专注做某件事情时，外界环境的微小噪音很容易打断我的注意力，让我分心。在外人看来，我似乎总在搞这搞那，没有专心在当下的任务中，效率很低。此外，我经常会忘记之前答应过别人的事情，给人留下不靠谱的印象。

上述三个问题——白日梦、易分心、不靠谱——对我的影响程度不尽相同。白日梦问题可以通过跟老师做好沟通来解决。我发现当我做自己感兴趣或有挑战性的任务时，我会表现出超专注。感谢 HFI 宽容又灵活的课程体系，让我得以在课程上保持新鲜与挑战。工作记忆缺失问题可以通过用好日程提醒工具来解决，老师也贴心地为我安排了一个”助理“，数字+人工双管齐下，能够确保我不会忘记重要日程。

最困扰我的是**易分心**问题，这不单纯是找个安静的环境就能解决的问题。现代神经生物学研究表明，ADHD 患者的大脑在基线状态下普遍处于“唤醒不足”的状态。在完全安静或单调的听觉环境中，ADHD 患者的神经系统会自发地在环境中寻求额外的外部感觉输入。我发现，适度的听觉刺激能够帮助我一定程度地屏蔽外界干扰，因此 Spotify Focus Playlist 对我来说是一个很好的工具。


## Focus Music 的特征

听觉上，Focus Music 与一般的商业音乐有着显著的差异。Focus Music 更加安静平缓，节奏感弱，且包含较少的人声；而一般的商业音乐为了吸引听众注意力，往往会更响亮，节奏感更强，情绪起伏更大。即便是平缓的抒情歌曲，也会试图通过歌词唤起听众情感共鸣。

我利用 Spotify API 获取了 Focus Music、Classical Music 和 Pop Music 的音频特征数据，定量地分析了这三个音乐类型的差异（详见《改善注意力缺陷的功能性音乐特征分析》）。通过对 `Instrumentalness`（器乐性）、`Danceability`（舞蹈度）、`Energy`（能量）、`Liveness`（现场度）、`Speechiness`（言语性）、`loudness`（响度）、`acousticness`（原生度）以及 `Valence`（情绪效价）的统计分析，我发现 Focus Music 具有如下特征：

- Focus Music 在能量（energy）、响度（loudness）、器乐性（instrumentalness）和原生性（acousticness）上的分布与 Pop Music 呈现上下颠倒的格局。Focus Music 整体表现为**低能量、低响度、高器乐性和高原生性**
- Focus Music 与 Classical Music 在能量、响度、器乐性和原生性上高度重叠，说明二者都属于安静、无人声、以真实乐器为主的音乐形态。然而，Focus Music 的情绪效价（valence）远低于 Classical Music，中位数仅为 $0.0655$，而古典音乐约为 $0.34$。这表明功能性专注音乐在情绪设计上刻意偏向“冷淡”“抽离”，避免了古典音乐中常见的情感起伏和叙事性张力。
- Focus Music 的 `danceability` 均值仅为 $0.31$，中位数低至 $0.22$，是所有类别中最低的。这意味着它没有强烈的节拍驱动感和身体律动引导，不会像流行音乐那样迫使听者进入节奏跟随状态。

基于以上分析结论，我们可以勾勒出适用于 ADHD 人群的理想 Focus Music 特征：

- 低能量（`energy` < $0.2$）
- 低响度（`loudness` < $-20$ dB）
- 极高器乐性（`instrumentalness` > $0.8$）
- 高原生性（`acousticness` > $0.8$）
- 低情绪效价（`valence` < $0.15$）
- 低舞蹈性（`danceability` < $0.30$）
- 节奏适中（`tempo` 约在 $90–110$ BPM区间）

## Focus Music 生成

我将基于上面分析得到的理想 Focus Music 特征，构建 prompt，利用 Suno v5.5 pro 和 ACE-Step V1.5 XL Turbo 两种模型生成 Focus Music，并对比二者的优缺点。选择 Suno 的原因是它是目前商业级音乐生成模型的 SOTA，而 ACE-Step 则是目前开源音乐生成模型的 SOTA。由于后面有计划对模型进行微调，因此需要关注开源模型的表现。

> 我很想测试腾讯 AI Lab 推出的 SongBloom 模型，但是 SongBloom 必须要输入一段参考音频，与我的应用场景不太相符。后面有时间我会单独进行测试。

### 指标性 prompt 构建

初次尝试，我决定先不构建复杂的提示词，而是直接利用上面分析得到的理想 Focus Music 特征，构建一个简单的指标性 prompt。

#### Simple 模式生成

两个工具在 Simple 模式下界面功能高度相似，都支持勾选 `Instrumental` 选项，用于生成纯音乐。因此，提示词中我们将去掉关于器乐性的描述。

| Suno  | ACE-Step |
| :---: | :---: |
| ![](img/suno01.png) | ![](img/ace-step01.png) <br/><br/><br/><br/>|

由于 `instrumentalness` 与 ``acousticness`` 高度正相关（详见《改善注意力缺陷的功能性音乐特征分析》第二部分“相关性分析”），为了减少特征维度，我们决定删除 `acousticness`。最终我们的提示词为

> ```text
> Generate a piece of functional music for improving concentration that meets the following feature requirements:
> 
> - Low energy (energy < 0.2)
> - Low loudness (loudness < -20 dB)
> - Low emotional valence (valence < 0.15)
> - Low danceability (danceability < 0.35)
> - Moderate tempo (tempo approximately in the range of 90–110 BPM)
> ```

Suno 和 ACE-Step 一次都会生成两首歌曲。下面是同一提示词下，二者生成的歌曲对比。

| Suno | ACE-Step |
| :---: | :---: |
| <audio src="audio/suno/Paper Rain Study by Fisher Yu_1.m4a" controls></audio><br><a href="https://suno.com/song/c5845cbb-ba9b-4b1e-9c12-908c513f8950">Suno #1: Paper Rain Study by Fisher Yu_1</a> | <audio src="audio/ace-step/Hand Claps in Motion.m4a" controls></audio><br><a href="https://acemusic.ai/playground/work?id=rg9MNArg">ACE-Step #1: Hand Claps in Motion</a> |
| <audio src="audio/suno/Paper Rain Study by Fisher Yu_2.m4a" controls></audio><br><a href="https://suno.com/song/fe67db0d-0556-4eeb-88bb-cab5b54f3abd">Suno #2: Paper Rain Study by Fisher Yu_2</a> | <audio src="audio/ace-step/Hypno Pulse.m4a" controls></audio><br><a href="https://acemusic.ai/playground/work?id=WBD3NrVj">ACE-Step #2: Hypno Pulse</a> |

从听觉上，Suno 生成的歌曲更符合 Focus Music 的特征，整体舒缓平静，没有明显的变化和情绪起伏。但两首歌都有明显的鼓声，节奏强烈。尤其是第二首，20 秒时导入的鼓声很突然，至少对我而言，这里吸引了我的注意力。

而 ACE-Step 生成的歌曲却令人失望。它的节奏快速，充斥着节奏单一的鼓点，给人一种紧张感，中间偶尔夹杂的管乐增加了恐怖的氛围，完全不符合 Focus Music 的特征。

我用 Essentia 提取了生成音频的特征（代码详见 `audio_features.py`），定量地分析生成音频与提示词指标的差异（代码详见 `plot.py`）。

<table border="1" cellspacing="0" cellpadding="10" width="100%">
    <tr>
        <th>Suno</th>
        <th>音频特征</th>
    </tr>
    <tr><td align="center">
<audio src="audio/suno/Paper Rain Study by Fisher Yu_1.m4a" controls></audio><br><a href="https://suno.com/song/c5845cbb-ba9b-4b1e-9c12-908c513f8950">Suno #1: Paper Rain Study by Fisher Yu_1</a>
</td> 
<td>

- bpm: 98
- danceability: 0.3957
- loudness: -14.5
- valence: 0.3451
- arousal: 0.4271
</td> 
</tr>
<tr><td align="center">
<audio src="audio/suno/Paper Rain Study by Fisher Yu_2.m4a" controls></audio><br><a href="https://suno.com/song/fe67db0d-0556-4eeb-88bb-cab5b54f3abd">Suno #2: Paper Rain Study by Fisher Yu_2</a>
</td> 
<td>

- bpm: 98
- danceability: 0.3806
- loudness: -14.4
- valence: 0.4084
- arousal: 0.4045
</td> 
</tr>
</table>

<table border="1" cellspacing="0" cellpadding="10" width="100%">
    <tr>
        <th>ACE-Step</th>
        <th>音频特征</th>
    </tr>
    <tr><td align="center">
<audio src="audio/ace-step/Hand Claps in Motion.m4a" controls></audio><br><a href="https://acemusic.ai/playground/work?id=rg9MNArg">ACE-Step #1: Hand Claps in Motion</a>
</td> 
<td>

- bpm: 124
- danceability: 0.5653
- loudness: -15.28
- valence: 0.5866
- arousal: 0.5521
</td> 
</tr>
<tr><td align="center">
<audio src="audio/ace-step/Hypno Pulse.m4a" controls></audio><br><a href="https://acemusic.ai/playground/work?id=WBD3NrVj">ACE-Step #2: Hypno Pulse</a>
</td> 
<td>

- bpm: 125
- danceability: 0.4938
- loudness: -16.54
- valence: 0.6151
- arousal: 0.5297
</td> 
</tr>
</table>

<style>
  .highlight-green {
    color: #000000;
    background-color: #D5E8D4;
  }
  .highlight-blue {
    color: #000000;
    background-color: #DAE8FC;
  }
</style>

<table border="1" cellspacing="0" cellpadding="10" width="100%">
    <tr>
        <th>音频特征</th>
        <th>Suno #1</th>
        <th>Suno #2</th>
        <th>ACE-Step #1</th>
        <th>ACE-Step #2</th>
    </tr>
    <tr>
        <th>bpm</th>
        <td class="highlight-green">98</td>
        <td class="highlight-green">98</td>
        <td>124 ↑</td>
        <td>125 ↑</td>
    </tr>
    <tr>
        <th>danceability</th>
        <td class="highlight-blue">0.3957 ~</td>
        <td class="highlight-blue">0.3806 ~</td>
        <td>0.5653 ↑</td>
        <td>0.4938 ↑</td>
    </tr>
    <tr>
        <th>loudness</th>
        <td>-14.50 ↑</td>
        <td>-14.40 ↑</td>
        <td>-15.28 ↑</td>
        <td>-16.54 ↑</td>
    </tr>
    <tr>
        <th>valence</th>
        <td>0.3451 ↑</td>
        <td>0.4084 ↑</td>
        <td>0.5866 ↑↑</td>
        <td>0.6151 ↑↑</td>
    </tr>
    <tr>
        <th>arousal</th>
        <td>0.4271</td>
        <td>0.4045</td>
        <td>0.5521</td>
        <td>0.5297</td>
    </tr>
</table> 

![](img/compare01.png)

从上表的数据对比可以看出，Suno 生成的音乐在 bpm 上满足提示词要求，danceability 与要求接近。loudness 比要求的高，不过这可以通过后期处理调节，不是关键评判因素。valence 高出很多，我认为这是由算法差异导致的。相比之下 ACE-Step 生成的音乐，各项数据都显著高于 Suno。ACE-Step 生成的音乐节奏更快、舞蹈性很高，情感也更强烈。完全不符合 Focus Music 的特征要求。

> **注意**
>
> 这里提取到的音频特征可能与 Spotify 提供的音频特征存在算法上的差异！例如 Spotify 中的 `energy` 是感知心理声学特征，衡量音乐带给听众的活跃度与动感，而 Essentia 中的 `energy` 是物理特征，反映音频信号的客观强弱。Spotify 的 `energy` 是一个黑盒专有模型，我没有找到近似替代算法，因此在特征对比中去掉了 `energy`。
>
> 此外 `danceability` 和 `valence` 的算法也存在差异！Essentia 的 `danceability` 的返回值在 0 到 ~3 之间，我将其归一化到 0 到 1 之间，从而与 Spotify 的 `danceability` 对齐。
>
> Essentia 的 `valence` 是通过 [DEAM](https://cvml.unige.ch/databases/DEAM/) 数据集在 musicnn 上预训练出的模型预测的，返回值 [1,9]，我也对其做了归一化处理，与 Spotify 的 `valence` 对齐。
>
> 上面提到的几个特征可能无法直接与提示词中的指标进行对比，但可以横向比较 Suno 和 ACE-Step 生成的音乐的特征差异。
>

由于 ACE-Step 是混合架构，其自带的大语言模型 acestep-5Hz-lm 负责将提示词转换为歌曲的全面规划。我在想，是否是因为 Simple 模式下没有开启 LM（或 LM 的思考模式）导致 ACE-Step 完全没有理解提示词中的指标。为了验证这一点，我用 Advanced 模式重新生成了一遍。

#### Advanced 模式生成

| Suno  | ACE-Step |
| :---: | :---: |
| ![](img/suno02.png) | ![](img/ace-step02.png) |

从界面可以明显看出，Advanced 模式下，Suno 和 ACE-Step 的控制参数复杂了很多。

首先，二者在 Advanced 模式下都去掉了 `Instrumental` 选项，我们可以在 Lyrics 中输入 `[Instrumental]` 控制模型生成纯音乐。

Prompt 依然沿用之前的指标性提示词，但配合二者各自的参数做了些许调整。

- **Suno**：由于上一次生成的音乐还是包含带有强烈节奏的鼓声，因此我在提示词中添加了 `No Percussion` 选项，用于禁用打击乐生成。其他参数保持系统默认。
- **ACE-Step**：ACE-Step 的控制参数比 Suno 细致，它可以控制生成的时长、节奏、拍号、曲调等参数，还可以输入负向提示词。我将时长设置为 60 秒，节奏设置为 100 BPM，去掉了提示词中关于节奏的描述。负向提示词设置为 `No Percussion`。同时开启 Thinking 模式，将 temperature 设置为 2.5，更偏向 Robust，以确保模型能够理解提示词中的指标。

调整后的提示词如下：

<table border="1" cellspacing="0" cellpadding="10" width="100%">
    <tr>
        <th>Suno</th>
        <th>ACE-Step</th>
    </tr>
    <tr><td style="vertical-align: top;">
Generate a piece of functional music for improving concentration that meets the following feature requirements:

- No Percussion
- Low energy (energy < 0.2)
- Low loudness (loudness < -20 dB)
- Low emotional valence (valence < 0.15)
- Low danceability (danceability < 0.30)
- Moderate tempo (tempo approximately in the range of 90–110 BPM)
</td> 
<td style="vertical-align: top;">
Generate a piece of functional music for improving concentration that meets the following feature requirements:

- Low energy (energy < 0.2)
- Low loudness (loudness < -20 dB)
- Low emotional valence (valence < 0.15)
- Low danceability (danceability < 0.30) 
</td> 
</tr>
</table>



生成结果如下：

<table border="1" cellspacing="0" cellpadding="10" width="100%">
    <tr>
        <th>Suno</th>
        <th>音频特征</th>
    </tr>
    <tr><td align="center">
<audio src="audio/suno/Tuning Forks Unwound by Fisher Yu_1.m4a" controls></audio><br><a href="https://suno.com/song/248cf8d5-c50c-4a8f-b73a-12dfa4f3358a">Suno #3: Tuning Forks Unwound by Fisher Yu_1</a>
</td> 
<td>

- bpm: 108
- danceability: 0.4127
- loudness: -14.32
- valence: 0.515
- arousal: 0.5018
</td> 
</tr>
<tr><td align="center">
<audio src="audio/suno/Tuning Forks Unwound by Fisher Yu_2.m4a" controls></audio><br><a href="https://suno.com/song/be942e81-f3dd-45af-9c04-c9e585d48b6e">Suno #4: Tuning Forks Unwound by Fisher Yu_2</a>
</td> 
<td>

- bpm: 110
- danceability: 0.3862
- loudness: -13.22
- valence: 0.569
- arousal: 0.5776
</td> 
</tr>
</table>

<table border="1" cellspacing="0" cellpadding="10" width="100%">
    <tr>
        <th>ACE-Step</th>
        <th>音频特征</th>
    </tr>
    <tr><td align="center">
<audio src="audio/ace-step/concentrate low energy.m4a" controls></audio><br><a href="https://acemusic.ai/playground/work?id=vBowZnKB">ACE-Step #3: Concentrate Low Energy</a>
</td> 
<td>

- bpm: 100
- danceability: 0.4007
- loudness: -13.49
- valence: 0.6586
- arousal: 0.6311
</td> 
</tr>
<tr><td align="center">
<audio src="audio/ace-step/concentrate on low energy.m4a" controls></audio><br><a href="https://acemusic.ai/playground/work?id=JBLQLdvm">ACE-Step #4: Concentrate on Low Energy</a>
</td> 
<td>

- bpm: 100
- danceability: 0.5454
- loudness: -11.92
- valence: 0.6701
- arousal: 0.6608
</td> 
</tr>
</table>

<style>
  .highlight-green {
    color: #000000;
    background-color: #D5E8D4;
  }
  .highlight-blue {
    color: #000000;
    background-color: #DAE8FC;
  }
</style>

<table border="1" cellspacing="0" cellpadding="10" width="100%">
    <tr>
        <th>音频特征</th>
        <th>Suno #3</th>
        <th>Suno #4</th>
        <th>ACE-Step #3</th>
        <th>ACE-Step #4</th>
    </tr>
    <tr>
        <th>bpm</th>
        <td>108 ↑</td>
        <td>110 ↑</td>
        <td>100</td>
        <td>100</td>
    </tr>
    <tr>
        <th>danceability</th>
        <td>0.4127 ↑</td>
        <td class="highlight-blue">0.3862 ~</td>
        <td>0.4007 ↑</td>
        <td>0.5454 ↑</td>
    </tr>
    <tr>
        <th>loudness</th>
        <td>-14.32 ↑</td>
        <td>-13.22 ↑</td>
        <td>-13.49 ↑</td>
        <td>-11.92 ↑</td>
    </tr>
    <tr>
        <th>valence</th>
        <td>0.5150 ↑↑</td>
        <td>0.5690 ↑↑</td>
        <td>0.6586 ↑↑</td>
        <td>0.6701 ↑↑</td>
    </tr>
    <tr>
        <th>arousal</th>
        <td>0.5018</td>
        <td>0.5776</td>
        <td>0.6311</td>
        <td>0.6608</td>
    </tr>
</table> 

![](img/compare02.png)

令人意想不到的是，Advanced 模式下，两个模型的生成的音乐居然全线崩坏。这否定了前面的假设，ACE-Step 确实无法理解音乐特征指标，这可能与模型训练数据的标注有关。值得注意的是，Ace-Step 很好得遵循了节奏要求（100 BPM），但负向提示词似乎没有其作用，生成的音乐中还是有明显的鼓点和打击乐。

Suno 输出质量的下降可能与其功能设计有关。在 Advanced 模式下，Suno 接受的不再是 prompt，而是 styles。Styles 是一系列描述音乐风格标签，例如 `melodramatic, shaker, glam pop, street punk, red dirt country` 等。当我输入指标性 prompt 到 styles 时，Suno 无法解析出有效的风格标签，从而导致输出质量下降。

经过上面指标性 prompt 实践，我们基本可以得出结论：

- Suno 的 Simple 模式对指标性 prompt 遵循性较好；
- ACE-Step 完全无法理解指标性 prompt。

下面，我将基于 Focus Music 的特征，构建语义性 prompt，测试 Suno 和 ACE-Step 对语义性 prompt 的遵循性。

### 语义性 prompt 构建

#### prompt v1.0

我们需要将 Focus Music 的特征映射到自然语言。这里我借助 ChatGPT 来完成提示词改写，然后再人工修正。

```text
nature ambient, around 100 BPM, drumless, no percussion, extremely low energy, quiet soundscape, detached mood, neutral valence, soft synth pads, sustained notes, flowing rhythm, zero groove, focus music, deep concentration
```

音频特征指标与语义提示词的对应关系如下：

| 音频特征指标 | 语义提示词 |
| --- | --- |
| No Percussion | `drumless, no percussion` |
| Low energy (energy < 0.2) | `extremely low energy, quiet soundscape` |
| Low emotional valence (valence < 0.15) | `detached mood, neutral valence` |
| Low danceability (danceability < 0.35) | `soft synth pads, sustained notes, flowing rhythm, zero groove` |
| Moderate tempo (tempo approximately in the range of 90–110 BPM) | `around 100 BPM` |
| 功能说明 | `focus music, deep concentration` |


我都采用 Simple 模式，生成结果如下：

<table border="1" cellspacing="0" cellpadding="10" width="100%">
    <tr>
        <th>Suno</th>
        <th>音频特征</th>
    </tr>
    <tr><td align="center">
<audio src="audio/suno/Drift Under Canopy by Fisher Yu_1.m4a" controls></audio><br><a href="https://suno.com/song/eb5adb29-6a2e-4f62-aad9-94c944297cde">Suno #5: Drift Under Canopy by Fisher Yu_1</a>
</td> 
<td>

- bpm: 100
- danceability: 0.3165
- loudness: -14.47
- valence: 0.2894
- arousal: 0.3117
</td> 
</tr>
<tr><td align="center">
<audio src="audio/suno/Drift Under Canopy by Fisher Yu_2.m4a" controls></audio><br><a href="https://suno.com/song/72a4ee30-5046-4758-8bd8-a473801431f5">Suno #6: Drift Under Canopy by Fisher Yu_2</a>
</td> 
<td>

- bpm: 100
- danceability: 0.3102
- loudness: -15.05
- valence: 0.3038
- arousal: 0.347
</td> 
</tr>
</table>

<table border="1" cellspacing="0" cellpadding="10" width="100%">
    <tr>
        <th>ACE-Step</th>
        <th>音频特征</th>
    </tr>
    <tr><td align="center">
<audio src="audio/ace-step/metallic pulse.m4a" controls></audio><br><a href="https://acemusic.ai/playground/work?id=7BWG2rlB">ACE-Step #5: metallic pulse</a>
</td> 
<td>

- bpm: 100
- danceability: 0.4847
- loudness: -17.36
- valence: 0.4638
- arousal: 0.356
</td> 
</tr>
<tr><td align="center">
<audio src="audio/ace-step/metallic resonance.m4a" controls></audio><br><a href="https://acemusic.ai/playground/work?id=dmyayp6m">ACE-Step #6: metallic resonance</a>
</td> 
<td>

- bpm: 100
- danceability: 0.3928
- loudness: -16.98
- valence: 0.4892
- arousal: 0.3758
</td> 
</tr>
</table>

<style>
  .highlight-green {
    color: #000000;
    background-color: #D5E8D4;
  }
  .highlight-blue {
    color: #000000;
    background-color: #DAE8FC;
  }
</style>

<table border="1" cellspacing="0" cellpadding="10" width="100%">
    <tr>
        <th>音频特征</th>
        <th>Suno #5</th>
        <th>Suno #6</th>
        <th>ACE-Step #5</th>
        <th>ACE-Step #6</th>
    </tr>
    <tr>
        <th>bpm</th>
        <td class="highlight-green">100</td>
        <td class="highlight-green">100</td>
        <td class="highlight-green">100</td>
        <td class="highlight-green">100</td>
    </tr>
    <tr>
        <th>danceability</th>
        <td class="highlight-green">0.3165</td>
        <td class="highlight-green">0.3102</td>
        <td>0.4847 ↑↑</td>
        <td class="highlight-blue">0.3928 ~</td>
    </tr>
    <tr>
        <th>loudness</th>
        <td>-14.47 ↑</td>
        <td>-15.05 ↑</td>
        <td>-17.36 ↑</td>
        <td>-16.98 ↑</td>
    </tr>
    <tr>
        <th>valence</th>
        <td>0.2894 ↑</td>
        <td>0.3038 ↑</td>
        <td>0.4638 ↑↑</td>
        <td>0.4892 ↑↑</td>
    </tr>
    <tr>
        <th>arousal</th>
        <td>0.3117</td>
        <td>0.347</td>
        <td>0.356</td>
        <td>0.3758</td>
    </tr>
</table> 

![](img/compare03.png)

无论是听觉上还是数据上，Suno 生成的音乐都明显好于 ACE-Step 生成的音乐。Suno 生成的音乐在听觉上更接近于真实音乐，而 ACE-Step 生成的音乐则很单一。氛围上，Suno 生成的音乐空灵安静，舒缓平和，而 ACE-Step 生成的音乐节奏感还是太强，这点在数据上也有直观的体现，Suno 生成的音乐在 `danceability` 和 `valence` 上数值显著低于 ACE-Step 生成的音乐。

Suno 生成的音乐也是存在瑕疵的，它并没有完全遵从 `drumless, no percussion` 的要求，而是包含了稀疏的低沉打击乐声，尤其是第一首音乐，在 42 秒左右，出现了比较明显的带节奏的鼓声。ACE-Step 生成的音乐确实没有鼓声，不过它用节奏强烈的弦乐（类似吉他的拍弦）代替了鼓声，同样给音乐带来了强烈的节奏感。

#### prompt v2.0

根据第一版提示词的输出质量，我们进一步优化了提示词，优化方向包括：

1. **增加原声性要求**

    Focus Music 具有很高的原声性（中位数 $0.925$），之前分析中未关注 `acousticness` 这个指标，但我们认为这是一个重要的特征，需要将其加入提示词中。我们向提示词中增加了 `acoustic music`，同时将 `soft synth pads` 修改为 `soft acoustic pads`，以减少音乐中电子元素的占比。

2. **进一步降低 danceability**

    打击乐是音乐中最主要的节奏感元素，我们认为消除打击乐元素可以进一步降低音乐的 danceability。尽管提示词中已经包含了 `drumless, no percussion`，但一首真实又丰富的音乐还是需要一定节奏支撑的，这可能是 Suno 会向音乐中加入打击乐元素的原因。我们决定进一步弱化对节奏感的要求，将 `flowing rhythm` 修改为 `weak rhythm`。

3. **增加 Lo-Fi 元素要求**

    我发现 Spotify Focus Playlist 中有大量 Lo-Fi 音乐，这些音乐刻意保留音质上的瑕疵，营造出温暖、怀旧且轻松的聆听体验。我们决定将 `Lo-Fi music` 作为音乐风格加入提示词中。

4. **降低 tempo**

    为了让生成的音乐更加舒缓，我们将 tempo 降低到 Focus Music tempo 的中位数 $96$ BPM。

优化后的提示词如下：
```text
nature ambient, 96 BPM, no drums, no percussion, extremely low energy, quiet soundscape, detached mood, neutral valence, soft acoustic pads, sustained notes, weak rhythm, zero groove, focus music, deep concentration, Lo-Fi music, acoustic music
```
    

为了更好地控制输出，本次采用 Advanced 模式生成。

| Suno  | ACE-Step |
| :---: | :---: |
| ![](img/suno03.png) | ![](img/ace-step03.png) |

为了进一步消除鼓点和打击乐，在 Suno 和 ACE-Step 的负向提示词中都加入了 `percussion, drums`。同时为了让模型更好地遵从提示词，Suno 的 Style Influence 我调大到了 80%，并且将 Weirdness 调低到 30%，降低生成音乐的惊奇度。ACE-Step 的 Thinking Level 我调大到了 2.7，使其更加稳定。

本次生成的结果，ACE-Step 有了巨大的提升，下面是生成的音乐和其特征：

<table border="1" cellspacing="0" cellpadding="10" width="100%">
    <tr>
        <th>Suno</th>
        <th>音频特征</th>
    </tr>
    <tr><td align="center">
<audio src="audio/suno/Moss-Tarp Quiet by Fisher Yu_1.m4a" controls></audio><br><a href="https://suno.com/song/2776672f-c95c-4386-8a56-677e5ca8db48?sh=ZKPoHj7BCB9KGpp5">Suno #7: Moss-Tarp Quiet by Fisher Yu_1</a>
</td> 
<td>

- bpm: 95
- danceability: 0.3544
- loudness: -14.23
- valence: 0.2875
- arousal: 0.2898
</td> 
</tr>
<tr><td align="center">
<audio src="audio/suno/Moss-Tarp Quiet by Fisher Yu_2.m4a" controls></audio><br><a href="https://suno.com/song/87056291-b3d0-4d7e-88dd-b89166f3ea90">Suno #8: Moss-Tarp Quiet by Fisher Yu_2</a>
</td> 
<td>

- bpm: 96
- danceability: 0.3595
- loudness: -13.80
- valence: 0.2822
- arousal: 0.3205
</td> 
</tr>
</table>

<table border="1" cellspacing="0" cellpadding="10" width="100%">
    <tr>
        <th>ACE-Step</th>
        <th>音频特征</th>
    </tr>
    <tr><td align="center">
<audio src="audio/ace-step/Nature's Quiet Pulse.m4a" controls></audio><br><a href="https://acemusic.ai/playground/work?id=lgk97aVB">ACE-Step #7: Nature's Quiet Pulse</a>
</td> 
<td>

- bpm: 96
- danceability: 0.3581
- loudness: -14.54
- valence: 0.3339
- arousal: 0.3056
</td> 
</tr>
<tr><td align="center">
<audio src="audio/ace-step/Nature's Quiet Pulse_2.m4a" controls></audio><br><a href="https://acemusic.ai/playground/work?id=nBbLRaZg">ACE-Step #8: Nature's Quiet Pulse_2</a>
</td> 
<td>

- bpm: 96
- danceability: 0.3484
- loudness: -16.43
- valence: 0.2964
- arousal: 0.2775
</td> 
</tr>
</table>

<style>
  .highlight-green {
    color: #000000;
    background-color: #D5E8D4;
  }
  .highlight-blue {
    color: #000000;
    background-color: #DAE8FC;
  }
</style>

<table border="1" cellspacing="0" cellpadding="10" width="100%">
    <tr>
        <th>音频特征</th>
        <th>Suno #7</th>
        <th>Suno #8</th>
        <th>ACE-Step #7</th>
        <th>ACE-Step #8</th>
    </tr>
    <tr>
        <th>bpm</th>
        <td class="highlight-green">95</td>
        <td class="highlight-green">96</td>
        <td class="highlight-green">96</td>
        <td class="highlight-green">96</td>
    </tr>
    <tr>
        <th>danceability</th>
        <td class="highlight-blue">0.3544 ~</td>
        <td class="highlight-blue">0.3595 ~</td>
        <td class="highlight-blue">0.3581 ~</td>
        <td class="highlight-blue">0.3484 ~</td>
    </tr>
    <tr>
        <th>loudness</th>
        <td>-14.23 ↑</td>
        <td>-13.80 ↑</td>
        <td>-14.54 ↑</td>
        <td>-16.43 ↑</td>
    </tr>
    <tr>
        <th>valence</th>
        <td>0.2875 ↑</td>
        <td>0.2822 ↑</td>
        <td>0.3339 ↑</td>
        <td>0.2964 ↑</td>
    </tr>
    <tr>
        <th>arousal</th>
        <td>0.2898</td>
        <td>0.3205</td>
        <td>0.3056</td>
        <td>0.2775</td>
    </tr>
</table> 

![](img/compare04.png)

从数值上看，Suno 和 Ace-Step 在音频特征上已经非常接近了，听觉上，我个人认为 Suno 生成的音乐会更温暖更饱满一些，而 Ace-Step 生成的音乐以吉他导入后面跟清脆的钢琴声，这种变化容易打破专注。并且高音区钢琴声穿透力很强，我在离得较远的地方也能听得到，这对 Focus Music 来说是不合适的。也许可以通过提示词对此加以控制。

Suno 生成的音乐依然顽固地加入鼓声，鼓声的加入一定程度上拉高了音乐的 danceability。如何消除鼓声是一个值得进一步研究的问题。

## 消除鼓声

关于如何让音乐大模型遵循指令不要生成鼓声，我查阅了一些文章。[Why AI models Ignores Your "No Drums" Prompt (And What Actually Works)](https://neuralanalog.com/docs/remove-drums-suno-prompt-issue) 这篇文章中提到，AI 音乐中普遍存在“粉红大象”现象——即通过文本提示移除鼓声，就像告诉别人“不要去想粉红大象”一样，这个指令不会令模型输出更好，反而会使问题变得更糟。文中还指出：**Stem 分离是去除鼓声的唯一有效方法**。

Suno 支持 Stem 分离，我尝试分离了 Suno 用 v2.0 提示词生成的两首歌（[Suno #7](https://suno.com/song/2776672f-c95c-4386-8a56-677e5ca8db48) 和 [Suno #8](https://suno.com/song/87056291-b3d0-4d7e-88dd-b89166f3ea90)）。

![](img/suno04.png)

Suno #7 分离出鼓、吉他、贝斯和其他垫音，将分离的音轨下载下来后，剔除鼓音轨，用 ffmpeg 再将他们混合，从而得到无鼓版本。

```bash
cd stems/suno7
ffmpeg -i Guitar.wav -i Bass.wav -i FX.wav -filter_complex "[0:a]volume=1.0[a1];[1:a]volume=1.0[a2];[2:a]volume=1.0[a3];[a1][a2][a3]amix=inputs=3:duration=longest:normalize=0" -f mp3 suno7.mp3
```

用同样的方法处理 Suno #8，得到 Suno #8 的无鼓版本。

```bash
cd stems/suno8
ffmpeg -i Guitar.wav -i Bass.wav -i Keyboard.wav -i Synth.wav -filter_complex "[0:a]volume=1.0[a1];[1:a]volume=1.0[a2];[2:a]volume=1.0[a3];[3:a]volume=1.0[a4];[a1][a2][a3][a4]amix=inputs=4:duration=longest:normalize=0" -f mp3 suno8.mp3
```

<table border="1" cellspacing="0" cellpadding="10" width="100%">
    <tr>
        <th>无鼓版本</th>
        <th>音频特征</th>
    </tr>
    <tr><td align="center">
<audio src="stems/suno7/suno7.mp3" controls></audio><br>Suno #7 无鼓版本
</td> 
<td>

- bpm: 95
- danceability: 0.2965
- loudness: -18.0
- valence: 0.2853
- arousal: 0.242
</td> 
</tr>
<tr><td align="center">
<audio src="stems/suno8/suno8.mp3" controls></audio><br>Suno #8 无鼓版本
</td> 
<td>

- bpm: 96
- danceability: 0.2879
- loudness: -18.3
- valence: 0.2937
- arousal: 0.2584
</td> 
</tr>
</table>

![](img/compare05.png)

去掉鼓声后，Suno #7 的 `danceability` 由 0.3544 降低到了 0.2965，下降了 16%;Suno #8 的 `danceability` 由 0.3595 降低到了 0.2937，下降了 18%;`valence`, `loudness` 和 `arousal` 都有所下降。可见消除鼓声对 Focus Music 的影响较大。

## 总结

|对比维度	| Suno	 | ACE-Step |
|-------   |-----   | -------- |
|指标性 prompt 遵循性 |	较好（Simple 模式）	| 完全无法理解 |
|语义性 prompt 遵循性 |	优秀，生成音乐空灵、舒缓<br><small>（Advanced 模式, Weirdnes=30%, Style Influence=80%）</small> | 有提升，但高音区易打破专注<br><small>（Advanced 模式, thinking=2.7）</small> |
|音频特征符合度 |	bpm、danceability、valence 控制较好	| bpm 控制准确，其他特征偏高 |
|音乐丰富度 |	更接近真实音乐，温暖饱满 | 相对单一 |
|鼓声消除 |	无法完全避免，需 Stem 分离 | 无鼓声但用拍弦替代 |
|适用性 |	适合直接生成高质量 Focus Music | 不太适合直接生成 Focus Music，需要有针对性地微调 |

最终结论：

- Suno 在生成 Focus Music 方面明显优于 ACE-Step，尤其是在语义性 prompt 下能产生更符合 ADHD 用户需求的平静、低舞蹈性、低情绪效价的音乐。

- ACE-Step 虽然在 v2.0 提示词下有所改善，但整体表现仍不稳定，且高音区乐器容易分散注意力。

- ACE-Step 的优势是开源，我们可以有针对性地微调模型，使其能够生成符合 ADHD 用户需求的音乐。

- 鼓声问题难以完全避免，当前最有效的解决方法是生成后进行 Stem 分离并剔除鼓声音轨。